In [8]:
from cvtypecorr.data.rxn import FormationReaction, ThermoReaction, RedoxReaction, AcidDeprotReaction


# defining a formation reaction

no2_formation = FormationReaction(
    "NO2", # For a formation reaction, give the molecule name
    [("N", 1), ("O", 2)], # the formula is a list of tuples of (element, count)
    -174.8, # Next give the experimental formation energy in ev/molecule
    el_by_priority=None, # In defining the formation reaction, reactants which represent the elements present
    # in the molecule are added until mass balance is achieved. If a reference specie contains elements not present
    # in the molecule, the reference species for those elements are added with negative coefficients.
    # ie, for NO2, the reference specie for N may be NH3 - since this will add 3H, we will see
    # the reference specie for H (H2) in our formation reaction.
    # This process will begin by enumerating all the elements present in the molecule ad the elements present
    # in the reference species for each of the molecule's elements. Then, in some order, the mass balance
    # is achieved element by element by adding enough of the reference specie for that element.
    # By default, the order is descending with respect to atomic number, (ie for NO2-, this will proceed
    # as O, N, then H), but this can be changed by defining el_by_priorty. If mass balance is not achieved
    # after going through the priorirty list, the reactants are modified by restarting the process.
)


# gives a reference reaction for the formation using the available reference species
# idk how to make the reference species customizable yet, but this is a start

print(no2_formation.reactants)

[('H2O', 2.0), ('NH3', 1.0), ('H2', -3.5)]


In [ ]:
# Defining a regular thermo reaction

c2h5oh_combustion = ThermoReaction(
    "C2H5OH + 3 O2 -> 2 CO2 + 3 H2O",
    [("C2H5OH", 1), ("O2", 3)],
    [("CO2", 2), ("H2O", 3)],
    -1367.0,
)



In [ ]:
# Defining a redox reaction

no3_to_no_redox = RedoxReaction(
    "NO3 + 3 e- -> NO + 2H2O",
    [("NO3-", 1), ("e-", 3), ("H3O+", 4)], # Must give electrons as a reactant for reduction, or as products for oxidation.
    [("NO", 1), ("H2O", 6)],
    0.96, # This is the standard reduction potential in V vs SHE.
    v0 = 4.66, # For non-CANDLE reactions, we can change this to another calibrated μ_SHE.
)

# a RedoxReaction will compute a delta_E by assuming the electrons have no energy.
print(f"Electron-less reaction energy with v0 = {no3_to_no_redox.v0}:", no3_to_no_redox.delta_E)

# Changing the reference 0V potential will change the electron-less reaction energy.
no3_to_no_redox.v0 = 4.55
print(f"Electron-less reaction energy with v0 = {no3_to_no_redox.v0}:", no3_to_no_redox.delta_E)


Electron-less reaction energy with v0 = 4.66: 11.100000000000001
Electron-less reaction energy with v0 = 4.55: 10.77


In [10]:
from cvtypecorr.data.pka import data_dict

# Defining an acid strength

acid = "NO3H"
base = "NO3-"
acid_pka = data_dict[acid]

no3h_to_no3_acid_deprot = AcidDeprotReaction(
    "NO3H -> NO3- + H+",
    acid,
    base,
    acid_pka,
    ref_acid_base_pair_and_pKa=("H3O+", "H2O", 0.0), # By default, hydronium is used as the reference counter acid-base pair, but this can be changed if desired.
    T=298.15, # Since conversion of pKa to free energy is temperature dependent, the temperature can be specified here. By default, it is 298.15 K.
)

print(f"Acid deprotonation reaction energy for {acid} with pKa = {acid_pka}:", no3h_to_no3_acid_deprot.delta_E)

Acid deprotonation reaction energy for NO3H with pKa = -1.3: -0.07690715459021705
